# Is the pipeline above chance, and do the designs differ?

- **Part A** (seconds, saved predictions only): AD vs CN AUC and balanced accuracy of the existing hybrid AD-vs-rest network on AD and CN
  subjects; a label-permutation test of each design's macro-F1 against chance; and the Nadeau-Bengio corrected t-test on the 15 per-fold
  macro-F1 differences between designs.
- **Part B**: a logistic regression trained only on AD and CN subjects (frozen features, same folds, same fitting subjects, threshold 0.5),
  the standard AD vs CN sanity check, for each backbone and for the backbone chosen on the calibration subjects.
- **Part C**: the design comparison without the feature confound. Multinomial, One-vs-All and two-stage designs built on the same frozen
  DenseNet-121 features, with the paper's calibration rules; and the One-vs-All hybrid from the runs with each network Platt-scaled on the
  calibration subjects before the argmax.
- **Part D**: is the leakage gain about fitting individual subjects? Logistic regression, 1-nearest-neighbour and an RBF-SVM on frozen
  DenseNet-121 features, under the slice split and the subject split, with true labels and with labels shuffled between subjects.

In [ ]:
import os, json, math, random, shutil, time, pickle, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image
from scipy import stats, ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (resnet18, ResNet18_Weights, densenet121, DenseNet121_Weights,
                                resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
INPUT_ROOT = '/kaggle/input'
WORK_DIR   = '/kaggle/working'
OUT_DIR = os.path.join(WORK_DIR, 'results_floor_checks')
os.makedirs(OUT_DIR, exist_ok=True)

IMAGE_SIZE  = 224
BATCH_SIZE  = 64
NUM_WORKERS = 2
N_PERM = 2000

SEED = 42
N_OUTER      = 5
CAL_FRACTION = 0.20
PRETRAINED   = True
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES = ['AD', 'CN', 'MCI']
AD, CN, MCI = 0, 1, 2
BACKBONES = ['resnet18+densenet121', 'resnet50', 'densenet121', 'efficientnet_b0']
PCA_DIM = 256
RANDAUG_N, RANDAUG_M = 2, 7
RANDOM_ERASING  = 0.20

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)

DESIGNS = {'ova': 'Hybrid, OvA', 'two_stage': 'Two-stage, tuned', 'two_stage_default': 'Two-stage, thresholds 0.5',
           'single_selected': 'Single-stage, selected'}

runs, run_dirs = {}, {}
for root, _, files in os.walk(INPUT_ROOT):
    name = os.path.basename(root)
    if name.startswith('results_patient_r') and 'patient_predictions.csv' in files:
        r = int(name.split('_r')[1])
        runs[r] = pd.read_csv(os.path.join(root, 'patient_predictions.csv'), dtype={'patient': str})
        run_dirs[r] = root
assert runs, 'Attach the outputs of the patient_r0, patient_r1 and patient_r2 runs of Team_09_TA_RepeatedCV'
print('Repeats found:', sorted(runs))

Repeats found: [0, 1, 2]


In [ ]:
def macro_f1(y, p):
    y, p = np.asarray(y), np.asarray(p)
    f = []
    for c in range(3):
        tp = np.sum((y == c) & (p == c))
        fp = np.sum((y != c) & (p == c))
        fn = np.sum((y == c) & (p != c))
        f.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0)
    return float(np.mean(f))

def mean_sd(v):
    v = np.asarray(v, dtype=float)
    return f'{v.mean():.3f} ± {v.std(ddof=1):.3f}' if len(v) > 1 else f'{v.mean():.3f}'

def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows():
        out.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
    return '\n'.join(out)

## Part A: saved predictions only

In [ ]:
def binary_ba(y_pos, score, t=0.5):
    y_pos, pred = np.asarray(y_pos, dtype=bool), np.asarray(score) >= t
    return float((pred[y_pos].mean() + (~pred[~y_pos]).mean()) / 2)

rng = np.random.default_rng(SEED)
hyb_auc, hyb_ba, perm_rows = [], [], []
for r, df in sorted(runs.items()):
    sub = df[df['true'].isin([AD, CN])]
    y_pos = (sub['true'] == AD).values
    hyb_auc.append(roc_auc_score(y_pos, sub['p_AD'].values))
    hyb_ba.append(binary_ba(y_pos, sub['p_AD'].values))
    y = df['true'].values
    for key, label in DESIGNS.items():
        pred = df[key].values
        obs = macro_f1(y, pred)
        null = np.array([macro_f1(rng.permutation(y), pred) for _ in range(N_PERM)])
        perm_rows.append({'design': label, 'repeat': r, 'macro-F1': round(obs, 3),
                          'null mean': round(float(null.mean()), 3), 'null 95th pct': round(float(np.percentile(null, 95)), 3),
                          'p': round(float((1 + np.sum(null >= obs)) / (1 + N_PERM)), 4)})
perm_df = pd.DataFrame(perm_rows)

fold_scores = defaultdict(list)
n_test, n_train = [], []
for r, df in sorted(runs.items()):
    for k, g in df.groupby('fold'):
        for key in DESIGNS:
            fold_scores[key].append(macro_f1(g['true'].values, g[key].values))
        n_test.append(len(g))
        n_train.append(len(df) - len(g))
J = len(n_test)
ratio = np.mean(n_test) / np.mean(n_train)
ttest_rows = []
for a, b in [('two_stage', 'ova'), ('two_stage', 'single_selected'), ('ova', 'single_selected')]:
    d = np.array(fold_scores[a]) - np.array(fold_scores[b])
    se = np.sqrt((1.0 / J + ratio) * d.var(ddof=1))
    t = d.mean() / se if se > 0 else 0.0
    ttest_rows.append({'comparison': f'{DESIGNS[a]} minus {DESIGNS[b]}', 'mean fold difference': round(float(d.mean()), 3),
                       't': round(float(t), 2), 'df': J - 1, 'p': round(float(2 * stats.t.sf(abs(t), J - 1)), 3)})
ttest_df = pd.DataFrame(ttest_rows)
fold_df = pd.DataFrame([{'design': DESIGNS[k], 'folds': len(v), 'fold macro-F1': mean_sd(v),
                         'min': round(min(v), 3), 'max': round(max(v), 3)} for k, v in fold_scores.items()])
print('Part A done')

Part A done


## Part B: logistic regression trained on AD vs CN only (frozen features, same folds)

In [ ]:
DATA_ROOT = None
for root, dirs, _ in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'adni_nifti_preprocessed' and {'AD', 'CN', 'MCI'} <= set(dirs):
        DATA_ROOT = root
        break
assert DATA_ROOT, 'Attach the ADNI dataset (a folder adni_nifti_preprocessed with AD, CN and MCI inside)'

patient_classes = defaultdict(set)
patient_files = defaultdict(list)
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    for pid in sorted(os.listdir(cls_dir)):
        pdir = os.path.join(cls_dir, pid)
        if not os.path.isdir(pdir):
            continue
        files = sorted(os.path.join(pdir, f) for f in os.listdir(pdir) if os.path.splitext(f)[1].lower() in VALID_EXTS)
        if files:
            patient_classes[pid].add(cls)
            patient_files[(pid, cls)] = files

patients = sorted(p for p, c in patient_classes.items() if len(c) == 1)
patient_label = {p: CLASS_NAMES.index(next(iter(patient_classes[p]))) for p in patients}

slice_path, slice_pid, slice_label = [], [], []
for p in patients:
    for f in patient_files[(p, CLASS_NAMES[patient_label[p]])]:
        slice_path.append(f)
        slice_pid.append(p)
        slice_label.append(patient_label[p])
slice_pid = np.array(slice_pid)
slice_label = np.array(slice_label)
n_slices = len(slice_path)

pid_idx = defaultdict(list)
for i, p in enumerate(slice_pid):
    pid_idx[p].append(i)
pid_idx = {p: np.array(v) for p, v in pid_idx.items()}

def idx_of(pids):
    return np.concatenate([pid_idx[p] for p in pids])

IMAGES = np.zeros((n_slices, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
for i, f in enumerate(slice_path):
    IMAGES[i] = np.array(Image.open(f).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS))

normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
hflip = transforms.RandomHorizontalFlip(p=0.5)
rand_aug = transforms.RandAugment(num_ops=RANDAUG_N, magnitude=RANDAUG_M)
jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15)
erasing = transforms.RandomErasing(p=RANDOM_ERASING, scale=(0.02, 0.12))

class SliceSet(Dataset):
    def __init__(self, idx, labels, augment):
        self.idx = np.asarray(idx)
        self.labels = np.asarray(labels)
        self.augment = augment

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        t = torch.from_numpy(IMAGES[self.idx[i]].copy()).unsqueeze(0).repeat(3, 1, 1)
        if self.augment:
            t = rand_aug(hflip(t))
            t = erasing(normalize(jitter(t.float() / 255.0)))
        else:
            t = normalize(t.float() / 255.0)
        return t, int(self.labels[i])

def make_loader(idx, labels, augment, shuffle):
    return DataLoader(SliceSet(idx, labels, augment), batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

counts = {c: sum(1 for p in patients if patient_label[p] == i) for i, c in enumerate(CLASS_NAMES)}
print(f'Subjects: {len(patients)} {counts} | slices: {n_slices}')

Subjects: 320 {'AD': 54, 'CN': 88, 'MCI': 178} | slices: 8512


In [ ]:
def build_extractor(name):
    if name == 'resnet18+densenet121':
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        class Both(nn.Module):
            def __init__(self):
                super().__init__()
                self.r, self.d = r, d
            def forward(self, x):
                return torch.cat([self.r(x), self.d(x)], dim=1)
        return Both()
    if name == 'resnet50':
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None)
        m.fc = nn.Identity()
        return m
    if name == 'densenet121':
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Identity()
        return m
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    m.classifier = nn.Identity()
    return m

feats = {}
loader = make_loader(np.arange(n_slices), np.zeros(n_slices, dtype=int), augment=False, shuffle=False)
for name in BACKBONES:
    ext = build_extractor(name).to(device).eval()
    out = []
    with torch.no_grad():
        for x, _ in loader:
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out.append(ext(x.to(device)).float().cpu().numpy())
    feats[name] = np.concatenate(out)
    del ext
    if device.type == 'cuda':
        torch.cuda.empty_cache()

def make_lr(n_fit):
    return make_pipeline(StandardScaler(), PCA(n_components=min(PCA_DIM, n_fit - 1), random_state=SEED),
                         LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
print('Features:', {k: v.shape[1] for k, v in feats.items()})

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 225MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 243MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 223MB/s]


Features: {'resnet18+densenet121': 1536, 'resnet50': 2048, 'densenet121': 1024, 'efficientnet_b0': 1280}


In [ ]:
def subject_scores(prob, idx):
    return pd.Series(prob).groupby(slice_pid[idx]).mean()

y_pat = np.array([patient_label[p] for p in patients])
folds_by_repeat = {}
for r, df in sorted(runs.items()):
    folds_by_repeat[r] = []
    skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=100 + r)
    for k, (tr, te) in enumerate(skf.split(np.zeros(len(patients)), y_pat)):
        tr_p = [patients[i] for i in tr]
        te_p = [patients[i] for i in te]
        fit_p, cal_p = train_test_split(tr_p, test_size=CAL_FRACTION, stratify=[patient_label[p] for p in tr_p],
                                        random_state=1000 + 10 * r + k)
        saved = set(df.loc[df['fold'] == k + 1, 'patient'])
        assert saved == set(te_p), f'repeat {r}, fold {k + 1}: the rebuilt test fold differs from the saved run'
        folds_by_repeat[r].append((fit_p, cal_p, te_p))

keys = BACKBONES + ['selected']
lr_auc, lr_ba = defaultdict(list), defaultdict(list)
sel_counts = defaultdict(int)
keep = lambda ps: [p for p in ps if patient_label[p] in (AD, CN)]
for r in sorted(runs):
    pooled = defaultdict(list)
    for k, (fit_p, cal_p, te_p) in enumerate(folds_by_repeat[r]):
        fit_i, cal_i, te_i = idx_of(keep(fit_p)), idx_of(keep(cal_p)), idx_of(keep(te_p))
        y_fit = (slice_label[fit_i] == AD).astype(int)
        cal_auc, te_sc = {}, {}
        for bb in BACKBONES:
            clf = make_lr(len(fit_i))
            clf.fit(feats[bb][fit_i], y_fit)
            cs = subject_scores(clf.predict_proba(feats[bb][cal_i])[:, 1], cal_i)
            cal_auc[bb] = roc_auc_score([patient_label[p] == AD for p in cs.index], cs.values)
            te_sc[bb] = subject_scores(clf.predict_proba(feats[bb][te_i])[:, 1], te_i)
            pooled[bb].append(te_sc[bb])
        best = max(BACKBONES, key=lambda b: cal_auc[b])
        sel_counts[best] += 1
        pooled['selected'].append(te_sc[best])
    for key in keys:
        s = pd.concat(pooled[key])
        y_pos = np.array([patient_label[p] == AD for p in s.index])
        lr_auc[key].append(roc_auc_score(y_pos, s.values))
        lr_ba[key].append(binary_ba(y_pos, s.values))

adcn_rows = [{'model': 'Hybrid AD-vs-rest network from the runs (no retraining)', 'AUC': mean_sd(hyb_auc), 'BA at 0.5': mean_sd(hyb_ba)}]
for key in keys:
    label = 'LR on AD vs CN, backbone chosen on calibration subjects' if key == 'selected' else f'LR on AD vs CN, frozen {key}'
    adcn_rows.append({'model': label, 'AUC': mean_sd(lr_auc[key]), 'BA at 0.5': mean_sd(lr_ba[key])})
adcn_df = pd.DataFrame(adcn_rows)

## Part C: designs on the same frozen features, and One-vs-All with calibrated scores

In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)
C_BACKBONE = 'densenet121'
OVA_TASKS = ['AD_vs_REST', 'CN_vs_REST', 'MCI_vs_REST']

def best_threshold(score_fn):
    best_t, best_s = 0.5, -1.0
    for t in THRESHOLDS:
        s = score_fn(t)
        if s > best_s + 1e-9 or (abs(s - best_s) <= 1e-9 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_s = float(t), s
    return best_t, best_s

def bin_macro_f1(y, p, t):
    pred = (np.asarray(p) >= t).astype(int)
    f = []
    for c in (0, 1):
        tp = np.sum((y == c) & (pred == c))
        fp = np.sum((y != c) & (pred == c))
        fn = np.sum((y == c) & (pred != c))
        f.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0)
    return float(np.mean(f))

def cascade(p_ad, p_mci, t1, t2):
    return np.where(p_ad >= t1, AD, np.where(p_mci >= t2, MCI, CN))

def subj_mean(prob, idx, order):
    df = pd.DataFrame(np.asarray(prob).reshape(len(idx), -1))
    df['pid'] = slice_pid[idx]
    return df.groupby('pid').mean().loc[order].values

def summarize(per_repeat):
    f1 = [macro_f1(y, p) for y, p in per_repeat]
    rec = np.array([[np.mean(p[y == c] == c) for c in range(3)] for y, p in per_repeat])
    return {'macro-F1': mean_sd(f1), 'BA': mean_sd(rec.mean(1)), 'R_AD': round(float(rec[:, AD].mean()), 2),
            'R_CN': round(float(rec[:, CN].mean()), 2), 'R_MCI': round(float(rec[:, MCI].mean()), 2),
            'predicted MCI': f'{np.mean([np.mean(p == MCI) for _, p in per_repeat]) * 100:.0f}%'}

X = feats[C_BACKBONE]
c_pooled = defaultdict(list)
agree = []
for r, df in sorted(runs.items()):
    ys, preds = [], defaultdict(list)
    for fit_p, cal_p, te_p in folds_by_repeat[r]:
        fit_i, cal_i, te_i = idx_of(fit_p), idx_of(cal_p), idx_of(te_p)
        cal_o, te_o = sorted(set(cal_p)), sorted(set(te_p))
        y_cal = np.array([patient_label[p] for p in cal_o])
        ys.append(np.array([patient_label[p] for p in te_o]))
        yf = slice_label[fit_i]
        multi = make_lr(len(fit_i)).fit(X[fit_i], yf)
        multi_te = subj_mean(multi.predict_proba(X[te_i]), te_i, te_o).argmax(1)
        preds['Multinomial (one three-class LR)'].append(multi_te)
        agree.append(np.mean(df.set_index('patient').loc[te_o, 'lr_' + C_BACKBONE].values == multi_te))
        ova_te, ova_cal = [], []
        for c in range(3):
            m = make_lr(len(fit_i)).fit(X[fit_i], (yf == c).astype(int))
            ova_te.append(subj_mean(m.predict_proba(X[te_i])[:, 1], te_i, te_o)[:, 0])
            ova_cal.append(subj_mean(m.predict_proba(X[cal_i])[:, 1], cal_i, cal_o)[:, 0])
        preds['One-vs-All (three binary LRs)'].append(np.stack(ova_te, 1).argmax(1))
        s2_i = fit_i[np.isin(yf, [CN, MCI])]
        s2 = make_lr(len(s2_i)).fit(X[s2_i], (slice_label[s2_i] == MCI).astype(int))
        p2_te = subj_mean(s2.predict_proba(X[te_i])[:, 1], te_i, te_o)[:, 0]
        p2_cal = subj_mean(s2.predict_proba(X[cal_i])[:, 1], cal_i, cal_o)[:, 0]
        nonad = y_cal != AD
        t2, _ = best_threshold(lambda t: bin_macro_f1((y_cal[nonad] == MCI).astype(int), p2_cal[nonad], t))
        t1, _ = best_threshold(lambda t: macro_f1(y_cal, cascade(ova_cal[AD], p2_cal, t, t2)))
        preds['Two-stage, thresholds tuned on calibration'].append(cascade(ova_te[AD], p2_te, t1, t2))
        preds['Two-stage, thresholds 0.5'].append(cascade(ova_te[AD], p2_te, 0.5, 0.5))
    y_all = np.concatenate(ys)
    for name, parts in preds.items():
        c_pooled[name].append((y_all, np.concatenate(parts)))
c_df = pd.DataFrame([dict({'design (frozen ' + C_BACKBONE + ' features)': name}, **summarize(v)) for name, v in c_pooled.items()])

# One-vs-All hybrid from the runs: each network's subject score Platt-scaled on the calibration subjects before the argmax
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

platt_pooled = defaultdict(list)
have_pkl = all(os.path.exists(os.path.join(run_dirs[r], f'fold{k + 1}.pkl')) for r in runs for k in range(N_OUTER))
if have_pkl:
    for r in sorted(runs):
        ys, raw, cal = [], [], []
        for k, (fit_p, cal_p, te_p) in enumerate(folds_by_repeat[r]):
            with open(os.path.join(run_dirs[r], f'fold{k + 1}.pkl'), 'rb') as fh:
                res = pickle.load(fh)
            cal_i, te_i = idx_of(cal_p), idx_of(te_p)
            assert np.array_equal(res['ev'], np.concatenate([cal_i, te_i])), f'repeat {r}, fold {k + 1}: saved slices do not match'
            n_cal = len(cal_i)
            cal_o, te_o = sorted(set(cal_p)), sorted(set(te_p))
            y_cal = np.array([patient_label[p] for p in cal_o])
            ys.append(np.array([patient_label[p] for p in te_o]))
            p_raw, p_cal = [], []
            for c, task in enumerate(OVA_TASKS):
                pc = subj_mean(res['hybrid'][task][:n_cal], cal_i, cal_o)[:, 0]
                pt = subj_mean(res['hybrid'][task][n_cal:], te_i, te_o)[:, 0]
                platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(pc)[:, None], (y_cal == c).astype(int))
                p_raw.append(pt)
                p_cal.append(platt.predict_proba(logit(pt)[:, None])[:, 1])
            raw.append(np.stack(p_raw, 1).argmax(1))
            cal.append(np.stack(p_cal, 1).argmax(1))
        y_all = np.concatenate(ys)
        platt_pooled['Hybrid OvA, as in the paper'].append((y_all, np.concatenate(raw)))
        platt_pooled['Hybrid OvA, Platt-scaled on calibration subjects'].append((y_all, np.concatenate(cal)))
    platt_df = pd.DataFrame([dict({'design': name}, **summarize(v)) for name, v in platt_pooled.items()])
else:
    platt_df = pd.DataFrame()
    print('fold*.pkl files not found in the attached run outputs, Platt scaling skipped')
print(c_df.to_string(index=False))
print(platt_df.to_string(index=False))

      design (frozen densenet121 features)      macro-F1            BA  R_AD  R_CN  R_MCI predicted MCI
          Multinomial (one three-class LR) 0.442 ± 0.021 0.446 ± 0.025  0.35  0.41   0.58           54%
             One-vs-All (three binary LRs) 0.440 ± 0.037 0.441 ± 0.038  0.32  0.41   0.59           56%
Two-stage, thresholds tuned on calibration 0.437 ± 0.018 0.442 ± 0.017  0.36  0.40   0.56           53%
                 Two-stage, thresholds 0.5 0.441 ± 0.031 0.441 ± 0.033  0.32  0.40   0.60           57%
                                          design      macro-F1            BA  R_AD  R_CN  R_MCI predicted MCI
                     Hybrid OvA, as in the paper 0.382 ± 0.018 0.380 ± 0.015  0.17  0.29   0.68           67%
Hybrid OvA, Platt-scaled on calibration subjects 0.284 ± 0.021 0.343 ± 0.015  0.04  0.05   0.94           94%


## Part D: leakage gain and the capacity to fit individual subjects

In [ ]:
D_BACKBONE = 'densenet121'
CAP_MODELS = ['LR (C = 0.1, as in the paper)', '1-nearest neighbour', 'RBF-SVM (C = 100)']

def make_cap(name, n_fit):
    pca = PCA(n_components=min(PCA_DIM, n_fit - 1), random_state=SEED)
    if name == CAP_MODELS[0]:
        clf = LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000)
    elif name == CAP_MODELS[1]:
        clf = KNeighborsClassifier(n_neighbors=1)
    else:
        clf = SVC(kernel='rbf', C=100.0, gamma='scale', class_weight='balanced')
    return make_pipeline(StandardScaler(), pca, clf)

r0 = min(runs)
subject_folds = [(idx_of(fit_p), idx_of(te_p)) for fit_p, _, te_p in folds_by_repeat[r0]]
slice_folds = []
for k, (tr, te) in enumerate(StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=200).split(np.zeros(n_slices), slice_label)):
    fit_i, _ = train_test_split(tr, test_size=CAL_FRACTION, stratify=slice_label[tr], random_state=2000 + k)
    slice_folds.append((np.sort(fit_i), np.sort(te)))

# shuffled labels: each subject gets another subject's label, so class counts stay the same and all slices of a subject share a label
perm = np.random.default_rng(SEED).permutation(len(patients))
shuffled = {p: patient_label[patients[j]] for p, j in zip(patients, perm)}
label_sets = {'true labels': slice_label, 'labels shuffled between subjects': np.array([shuffled[p] for p in slice_pid])}

X = feats[D_BACKBONE]
cap_rows = []
for lab_name, lab in label_sets.items():
    for model_name in CAP_MODELS:
        row = {'labels': lab_name, f'classifier (frozen {D_BACKBONE})': model_name}
        for split_name, folds in (('slice split', slice_folds), ('subject split', subject_folds)):
            yt, yp = [], []
            for fit_i, te_i in folds:
                clf = make_cap(model_name, len(fit_i)).fit(X[fit_i], lab[fit_i])
                yt.append(lab[te_i])
                yp.append(clf.predict(X[te_i]))
            row[split_name + ': slice F1'] = round(macro_f1(np.concatenate(yt), np.concatenate(yp)), 3)
        row['gain from the slice split'] = round(row['slice split: slice F1'] - row['subject split: slice F1'], 3)
        cap_rows.append(row)
cap_df = pd.DataFrame(cap_rows)
print(cap_df.to_string(index=False))

                          labels classifier (frozen densenet121)  slice split: slice F1  subject split: slice F1  gain from the slice split
                     true labels   LR (C = 0.1, as in the paper)                  0.647                    0.394                      0.253
                     true labels             1-nearest neighbour                  0.996                    0.379                      0.617
                     true labels               RBF-SVM (C = 100)                  0.989                    0.360                      0.629
labels shuffled between subjects   LR (C = 0.1, as in the paper)                  0.578                    0.315                      0.263
labels shuffled between subjects             1-nearest neighbour                  0.995                    0.324                      0.671
labels shuffled between subjects               RBF-SVM (C = 100)                  0.987                    0.315                      0.672


## Part E (optional): volumetric baseline from ADNIMERGE

In [ ]:
merge_path = None
for root, _, files in os.walk(INPUT_ROOT):
    for n in files:
        if n.lower().startswith('adnimerge') and n.lower().endswith('.csv'):
            merge_path = os.path.join(root, n)

vol_df, vol_note = pd.DataFrame(), 'ADNIMERGE.csv not attached, Part E skipped'
if merge_path:
    am = pd.read_csv(merge_path, low_memory=False)
    am = am[am['VISCODE'].astype(str) == 'bl'].drop_duplicates('PTID').set_index('PTID').reindex(patients)
    num = lambda c: pd.to_numeric(am[c], errors='coerce')
    V = pd.DataFrame(index=patients)
    for c in ['Hippocampus', 'Ventricles', 'WholeBrain']:
        V[c + '/ICV'] = num(c) / num('ICV')
    V['age'] = num('AGE')
    V['male'] = np.where(am['PTGENDER'].isna(), np.nan, (am['PTGENDER'] == 'Male').astype(float))
    have = V.notna().all(axis=1)
    covered = {c: int(sum(have[p] for p in patients if patient_label[p] == i)) for i, c in enumerate(CLASS_NAMES)}
    vol_note = f'Subjects with all baseline values in ADNIMERGE: {int(have.sum())} of {len(patients)} {covered}'
    feature_sets = {'LR on volumes/ICV + age + sex': list(V.columns), 'LR on age + sex only': ['age', 'male']}
    pooled, aucs = defaultdict(list), defaultdict(list)
    for r, df in sorted(runs.items()):
        ys, preds, ad_y, ad_s = [], defaultdict(list), [], defaultdict(list)
        for fit_p, cal_p, te_p in folds_by_repeat[r]:
            fit_v = [p for p in fit_p if have[p]]
            te_v = [p for p in te_p if have[p]]
            fit_b = [p for p in fit_v if patient_label[p] in (AD, CN)]
            te_b = [p for p in te_v if patient_label[p] in (AD, CN)]
            ys.append(np.array([patient_label[p] for p in te_v]))
            ad_y.append(np.array([patient_label[p] == AD for p in te_b]))
            for fs, cols in feature_sets.items():
                m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight='balanced', max_iter=3000))
                m.fit(V.loc[fit_v, cols].values, [patient_label[p] for p in fit_v])
                preds[fs].append(m.predict(V.loc[te_v, cols].values))
                mb = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight='balanced', max_iter=3000))
                mb.fit(V.loc[fit_b, cols].values, [patient_label[p] == AD for p in fit_b])
                ad_s[fs].append(mb.predict_proba(V.loc[te_b, cols].values)[:, 1])
            sub = df.set_index('patient')
            for key in ('ova', 'two_stage', 'single_selected'):
                preds['CNN: ' + DESIGNS[key]].append(sub.loc[te_v, key].values)
            ad_s['CNN: ' + DESIGNS['ova']].append(sub.loc[te_b, 'p_AD'].values)
        y_all, yad = np.concatenate(ys), np.concatenate(ad_y)
        for name, parts in preds.items():
            pooled[name].append((y_all, np.concatenate(parts)))
        for name, parts in ad_s.items():
            aucs[name].append(roc_auc_score(yad, np.concatenate(parts)))
    vol_df = pd.DataFrame([dict({'model (same subjects and folds)': name}, **summarize(v),
                                **{'AD vs CN AUC': mean_sd(aucs[name]) if name in aucs else ''}) for name, v in pooled.items()])
    print(vol_df.to_string(index=False))
print(vol_note)

ADNIMERGE.csv not attached, Part E skipped


## Results (saved to results_floor_checks)

In [ ]:
n_ad = int(sum(patient_label[p] == AD for p in patients))
n_cn = int(sum(patient_label[p] == CN for p in patients))
lines = [f'## AD vs CN at subject level ({n_ad} AD and {n_cn} CN subjects, mean ± SD over {len(runs)} repeats)', md_table(adcn_df), '',
         f'Backbone chosen on calibration subjects: {dict(sel_counts)}', '',
         f'## Permutation test against chance ({N_PERM} label permutations per repeat)', md_table(perm_df), '',
         f'## Corrected resampled t-test (Nadeau-Bengio), {J} folds, test/train ratio {ratio:.3f}', md_table(ttest_df), '',
         '## Per-fold macro-F1', md_table(fold_df), '',
         f'## Part C: designs on the same frozen {C_BACKBONE} features (mean ± spread over repeats)', md_table(c_df), '',
         f'Agreement of the multinomial LR with the saved lr_{C_BACKBONE} predictions: {np.mean(agree) * 100:.1f}% of subjects', '',
         '## Part C: One-vs-All hybrid with Platt scaling on the calibration subjects',
         md_table(platt_df) if len(platt_df) else 'skipped: fold*.pkl files not found in the run outputs', '',
         f'## Part D: slice-level macro-F1 under both splits (frozen {D_BACKBONE}; subject split uses the folds of repeat {r0})',
         md_table(cap_df), '',
         '## Part E: volumetric baseline from ADNIMERGE (baseline visit)', vol_note, '',
         md_table(vol_df) if len(vol_df) else '', '']
with open(os.path.join(OUT_DIR, 'floor_checks.md'), 'w') as fh:
    fh.write('\n'.join(lines))
with open(os.path.join(OUT_DIR, 'floor_checks.json'), 'w') as fh:
    json.dump({'ad_vs_cn': adcn_df.to_dict('records'), 'selected_backbones': dict(sel_counts),
               'permutation': perm_df.to_dict('records'), 'corrected_ttest': ttest_df.to_dict('records'),
               'per_fold': fold_df.to_dict('records'), 'n_ad': n_ad, 'n_cn': n_cn,
               'same_feature_designs': c_df.to_dict('records'), 'multinomial_agreement': float(np.mean(agree)),
               'platt_ova': platt_df.to_dict('records'), 'capacity': cap_df.to_dict('records'),
               'volumetric_note': vol_note, 'volumetric': vol_df.to_dict('records')}, fh, indent=1)
print('\n'.join(lines))

## AD vs CN at subject level (54 AD and 88 CN subjects, mean ± SD over 3 repeats)
| model | AUC | BA at 0.5 |
|---|---|---|
| Hybrid AD-vs-rest network from the runs (no retraining) | 0.716 ± 0.018 | 0.587 ± 0.020 |
| LR on AD vs CN, frozen resnet18+densenet121 | 0.776 ± 0.015 | 0.714 ± 0.028 |
| LR on AD vs CN, frozen resnet50 | 0.704 ± 0.046 | 0.610 ± 0.068 |
| LR on AD vs CN, frozen densenet121 | 0.783 ± 0.016 | 0.714 ± 0.027 |
| LR on AD vs CN, frozen efficientnet_b0 | 0.762 ± 0.002 | 0.685 ± 0.013 |
| LR on AD vs CN, backbone chosen on calibration subjects | 0.751 ± 0.002 | 0.703 ± 0.022 |

Backbone chosen on calibration subjects: {'efficientnet_b0': 9, 'resnet18+densenet121': 1, 'densenet121': 3, 'resnet50': 2}

## Permutation test against chance (2000 label permutations per repeat)
| design | repeat | macro-F1 | null mean | null 95th pct | p |
|---|---|---|---|---|---|
| Hybrid, OvA | 0 | 0.368 | 0.328 | 0.371 | 0.064 |
| Two-stage, tuned | 0 | 0.39 | 0.333 | 0.376 | 0.014 |
| T